# Índices — Estratégias de Indexação

Estratégias práticas de indexação no PostgreSQL: tipos de índice, composição, índices parciais e funcionais, diagnóstico de queries lentas e anti-patterns a evitar.

## Estrutura de um B-Tree

![Estrutura B-Tree no PostgreSQL](assets/02-btree-structure.png)

O B-Tree organiza os dados em nós: root, internos e folhas. As folhas contêm ponteiros (TIDs) para as linhas na tabela e são ligadas entre si — isso permite range scans eficientes sem voltar à raiz.

- **Igualdade** (`WHERE status = 'pendente'`): O(log n) — navega da raiz até a folha
- **Range** (`WHERE created_at BETWEEN A AND B`): O(log n) + scan sequencial nas folhas conectadas
- **ORDER BY**: as folhas já estão ordenadas, então o planner pode eliminar o nó Sort do plano

## Setup: Conexão e Schema

Schema do sistema de crédito: 10k clientes, 50k propostas (3% pendentes), 200k parcelas. Mesmo banco usado no notebook 01.

In [114]:
import psycopg2
import psycopg2.extras
import json
import random
from datetime import datetime, timedelta
from decimal import Decimal

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "fintech_kb",
    "user": "postgres",
    "password": "postgres",
}

conn = psycopg2.connect(**DB_CONFIG)
conn.autocommit = True
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

print("Conexão estabelecida.")

Conexão estabelecida.


In [115]:
# Criação do schema (DROP CASCADE para ambiente de estudo)
DDL = """
DROP TABLE IF EXISTS parcelas CASCADE;
DROP TABLE IF EXISTS decisoes_log CASCADE;
DROP TABLE IF EXISTS consultas_externas CASCADE;
DROP TABLE IF EXISTS propostas CASCADE;
DROP TABLE IF EXISTS clientes CASCADE;

CREATE TABLE clientes (
    id          SERIAL PRIMARY KEY,
    cpf         VARCHAR(11) NOT NULL UNIQUE,
    nome        VARCHAR(200) NOT NULL,
    renda_mensal NUMERIC(12, 2),
    created_at  TIMESTAMP NOT NULL DEFAULT NOW()
);

CREATE TABLE propostas (
    id          SERIAL PRIMARY KEY,
    numero      VARCHAR(20) NOT NULL UNIQUE,
    cliente_id  INT NOT NULL REFERENCES clientes(id),
    valor       NUMERIC(12, 2) NOT NULL,
    status      VARCHAR(20) NOT NULL DEFAULT 'pendente',
    score       INT,
    created_at  TIMESTAMP NOT NULL DEFAULT NOW()
);

CREATE TABLE consultas_externas (
    id          SERIAL PRIMARY KEY,
    proposta_id INT NOT NULL REFERENCES propostas(id),
    provider    VARCHAR(50) NOT NULL,
    response_data JSONB,
    status      VARCHAR(20) NOT NULL,
    created_at  TIMESTAMP NOT NULL DEFAULT NOW()
);

CREATE TABLE decisoes_log (
    id               SERIAL PRIMARY KEY,
    proposta_id      INT NOT NULL REFERENCES propostas(id),
    etapa            VARCHAR(50) NOT NULL,
    resultado        VARCHAR(20) NOT NULL,
    regras_aplicadas JSONB,
    executed_at      TIMESTAMP NOT NULL DEFAULT NOW()
);

CREATE TABLE parcelas (
    id               SERIAL PRIMARY KEY,
    proposta_id      INT NOT NULL REFERENCES propostas(id),
    numero           SMALLINT NOT NULL,
    valor            NUMERIC(10, 2) NOT NULL,
    vencimento       DATE NOT NULL,
    status_pagamento VARCHAR(20) NOT NULL DEFAULT 'pendente',
    pago_em          DATE
);
"""

cur.execute(DDL)
print("Schema criado.")

Schema criado.


In [116]:
# População: 10.000 clientes, 50.000 propostas, ~200.000 parcelas
# Maioria das propostas: aprovada/reprovada (dados frios)
# 3% pendente (dados quentes — alvo de índices parciais)

import string

STATUS_DIST = [
    ('aprovada',  0.45),
    ('reprovada', 0.35),
    ('cancelada', 0.10),
    ('analise',   0.07),
    ('pendente',  0.03),
]

def weighted_choice(dist):
    r = random.random()
    acc = 0.0
    for val, prob in dist:
        acc += prob
        if r < acc:
            return val
    return dist[-1][0]

# Clientes
clientes_data = [
    (
        str(random.randint(10000000000, 99999999999)),
        f"Cliente {i:05d}",
        round(random.uniform(1500, 25000), 2),
        datetime(2020, 1, 1) + timedelta(days=random.randint(0, 1800)),
    )
    for i in range(1, 10001)
]

psycopg2.extras.execute_values(
    cur,
    "INSERT INTO clientes (cpf, nome, renda_mensal, created_at) VALUES %s",
    clientes_data,
)
print(f"Clientes inseridos: {len(clientes_data)}")

# Propostas
propostas_data = [
    (
        f"PROP-{i:06d}",
        random.randint(1, 10000),
        round(random.uniform(500, 50000), 2),
        weighted_choice(STATUS_DIST),
        random.randint(300, 900),
        datetime(2022, 1, 1) + timedelta(days=random.randint(0, 1095)),
    )
    for i in range(1, 50001)
]

psycopg2.extras.execute_values(
    cur,
    "INSERT INTO propostas (numero, cliente_id, valor, status, score, created_at) VALUES %s",
    propostas_data,
)
print(f"Propostas inseridas: {len(propostas_data)}")

# Parcelas: 12 por proposta (aprovada ou pendente), ~200k linhas
cur.execute("""
    INSERT INTO parcelas (proposta_id, numero, valor, vencimento, status_pagamento, pago_em)
    SELECT p.id, n.numero, (p.valor / 12)::numeric(12,2),
        (NOW() - INTERVAL '10 months' + (n.numero * INTERVAL '1 month'))::date,
        CASE WHEN n.numero <= 6 THEN 'pago' ELSE 'pendente' END,
        CASE WHEN n.numero <= 6 THEN NOW() - (random() * INTERVAL '180 days') ELSE NULL END
    FROM propostas p
    CROSS JOIN generate_series(1, 12) AS n(numero)
    WHERE p.status = 'pendente' OR p.status = 'aprovada'
    LIMIT 200000;
""")
cur.execute("SELECT COUNT(*) AS total FROM parcelas")
parcelas_count = cur.fetchone()["total"]
print(f"Parcelas inseridas: {parcelas_count:,}")

# Distribuição de status
cur.execute("SELECT status, COUNT(*) FROM propostas GROUP BY status ORDER BY COUNT(*) DESC")
for row in cur.fetchall():
    print(f"  {row['status']:12s}: {row['count']:,}")

Clientes inseridos: 10000
Propostas inseridas: 50000
Parcelas inseridas: 200,000
  aprovada    : 22,473
  reprovada   : 17,547
  cancelada   : 5,019
  analise     : 3,509
  pendente    : 1,452


## 1. B-Tree Index — Quando e Como Usar

B-Tree é o default do PostgreSQL. Cobre a maioria dos casos.

Use B-Tree para:
- Igualdade: `WHERE status = 'aprovada'`
- Ranges e comparações: `BETWEEN`, `>`, `<`, `>=`
- ORDER BY sem custo extra de Sort (folhas já ordenadas)
- Prefix match em texto: `LIKE 'PROP-001%'` funciona, `LIKE '%texto%'` não

Onde B-Tree não serve:
- JSONB com `@>`, `?`, `?|` — precisa de GIN
- Wildcard no início do LIKE — precisa de pg_trgm ou full-text
- Colunas com 2-3 valores distintos em tabelas pequenas: o planner vai ignorar o índice e fazer Seq Scan porque o custo de random I/O não compensa

In [117]:
def explain(sql, params=None, analyze=False):
    """Executa EXPLAIN (ANALYZE, BUFFERS) e imprime o plano."""
    prefix = "EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)" if analyze else "EXPLAIN (FORMAT TEXT)"
    full_sql = f"{prefix} {sql}"
    cur.execute(full_sql, params)
    rows = cur.fetchall()
    for row in rows:
        print(row["QUERY PLAN"])

# Sem índice em status
print("=== SEM ÍNDICE ===")
explain("SELECT * FROM propostas WHERE status = 'aprovada'")

=== SEM ÍNDICE ===
Seq Scan on propostas  (cost=0.00..798.44 rows=115 width=152)
  Filter: ((status)::text = 'aprovada'::text)


In [118]:
# Criamos o índice B-Tree
cur.execute("CREATE INDEX IF NOT EXISTS idx_propostas_status ON propostas (status)")
print("Índice criado: idx_propostas_status")

print("\n=== COM ÍNDICE ===")
explain("SELECT * FROM propostas WHERE status = 'aprovada'")

Índice criado: idx_propostas_status

=== COM ÍNDICE ===
Bitmap Heap Scan on propostas  (cost=6.23..435.17 rows=250 width=152)
  Recheck Cond: ((status)::text = 'aprovada'::text)
  ->  Bitmap Index Scan on idx_propostas_status  (cost=0.00..6.17 rows=250 width=0)
        Index Cond: ((status)::text = 'aprovada'::text)


In [119]:
# Range query — B-Tree percorre folhas de forma sequencial entre os limites
SQL_RANGE = """
SELECT id, numero, valor
FROM propostas
WHERE created_at BETWEEN '2023-01-01' AND '2023-03-31'
"""

print("=== RANGE SEM ÍNDICE EM created_at ===")
explain(SQL_RANGE)

cur.execute("CREATE INDEX IF NOT EXISTS idx_propostas_created_at ON propostas (created_at)")

print("\n=== RANGE COM ÍNDICE EM created_at ===")
explain(SQL_RANGE)

=== RANGE SEM ÍNDICE EM created_at ===
Seq Scan on propostas  (cost=0.00..1261.00 rows=250 width=78)
  Filter: ((created_at >= '2023-01-01 00:00:00'::timestamp without time zone) AND (created_at <= '2023-03-31 00:00:00'::timestamp without time zone))

=== RANGE COM ÍNDICE EM created_at ===
Bitmap Heap Scan on propostas  (cost=6.85..436.42 rows=250 width=78)
  Recheck Cond: ((created_at >= '2023-01-01 00:00:00'::timestamp without time zone) AND (created_at <= '2023-03-31 00:00:00'::timestamp without time zone))
  ->  Bitmap Index Scan on idx_propostas_created_at  (cost=0.00..6.79 rows=250 width=0)
        Index Cond: ((created_at >= '2023-01-01 00:00:00'::timestamp without time zone) AND (created_at <= '2023-03-31 00:00:00'::timestamp without time zone))


## 2. Índice Composto — A Ordem das Colunas Importa

Num índice `(A, B)`, o PostgreSQL usa o índice para:
- Filtro em `A` sozinho
- Filtro em `A` + `B`

Filtro só em `B` (sem `A`): o índice não serve. Isso é a **leftmost prefix rule**.

**Ordem das colunas:**

Colunas de igualdade (`=`) primeiro, range (`BETWEEN`, `>`, `<`) ou `ORDER BY` depois. Motivo: igualdade restringe a busca a uma sub-árvore específica do B-Tree; range dentro dessa sub-árvore é barato. Se inverter, o range abre muitas sub-árvores e a igualdade filtra dentro de cada uma — mais I/O.

Exceção: se a coluna de igualdade tem cardinalidade muito baixa (ex: boolean) e a de range tem alta seletividade, a diferença desaparece. Teste com EXPLAIN ANALYZE.

In [120]:
# Removemos os índices simples para focar no composto
cur.execute("DROP INDEX IF EXISTS idx_propostas_status")
cur.execute("DROP INDEX IF EXISTS idx_propostas_created_at")

# Criamos dois índices compostos com ordens diferentes
cur.execute("""
    CREATE INDEX idx_comp_status_created
    ON propostas (status, created_at)
""")

cur.execute("""
    CREATE INDEX idx_comp_created_status
    ON propostas (created_at, status)
""")

print("Índices compostos criados.")

Índices compostos criados.


In [121]:
# Query 1: filtra por status (igualdade) + range de data
# Favorece (status, created_at) — igualdade primeiro reduz o espaço de busca
SQL_Q1 = """
SELECT id, numero
FROM propostas
WHERE status = 'aprovada'
  AND created_at >= '2023-01-01'
"""

print("=== Query: status = X AND created_at >= Y ===")
print("Esperado: idx_comp_status_created")
explain(SQL_Q1)

=== Query: status = X AND created_at >= Y ===
Esperado: idx_comp_status_created
Bitmap Heap Scan on propostas  (cost=5.14..224.72 rows=83 width=62)
  Recheck Cond: (((status)::text = 'aprovada'::text) AND (created_at >= '2023-01-01 00:00:00'::timestamp without time zone))
  ->  Bitmap Index Scan on idx_comp_status_created  (cost=0.00..5.12 rows=83 width=0)
        Index Cond: (((status)::text = 'aprovada'::text) AND (created_at >= '2023-01-01 00:00:00'::timestamp without time zone))


In [122]:
# Query 2: filtra só por created_at (sem status)
# Favorece (created_at, status) — leftmost prefix
# O índice (status, created_at) NÃO pode ser usado aqui sem status
SQL_Q2 = """
SELECT id, numero
FROM propostas
WHERE created_at >= '2023-06-01'
"""

print("=== Query: apenas created_at >= Y ===")
print("Esperado: idx_comp_created_status (ou Seq Scan se seletividade for baixa)")
explain(SQL_Q2)

=== Query: apenas created_at >= Y ===
Esperado: idx_comp_created_status (ou Seq Scan se seletividade for baixa)
Bitmap Heap Scan on propostas  (cost=225.46..944.80 rows=16667 width=62)
  Recheck Cond: (created_at >= '2023-06-01 00:00:00'::timestamp without time zone)
  ->  Bitmap Index Scan on idx_comp_created_status  (cost=0.00..221.29 rows=16667 width=0)
        Index Cond: (created_at >= '2023-06-01 00:00:00'::timestamp without time zone)


In [123]:
# Query 3: filtra só por status (sem created_at)
# O índice (status, created_at) pode ser usado — status é leftmost
SQL_Q3 = """
SELECT id, numero
FROM propostas
WHERE status = 'pendente'
"""

print("=== Query: apenas status = X ===")
print("Esperado: idx_comp_status_created (status é leftmost)")
explain(SQL_Q3)

=== Query: apenas status = X ===
Esperado: idx_comp_status_created (status é leftmost)
Bitmap Heap Scan on propostas  (cost=6.23..435.17 rows=250 width=62)
  Recheck Cond: ((status)::text = 'pendente'::text)
  ->  Bitmap Index Scan on idx_comp_status_created  (cost=0.00..6.17 rows=250 width=0)
        Index Cond: ((status)::text = 'pendente'::text)


## 3. Índice Parcial — Indexar Apenas o Subconjunto Relevante

Um índice parcial inclui apenas as linhas que satisfazem uma condição `WHERE`. Vantagens:

- **Menor tamanho em disco**: só indexa dados quentes
- **Menor custo de manutenção**: INSERTs/UPDATEs fora do predicado não tocam o índice
- **Mais eficiente**: índice menor cabe melhor em cache

Caso clássico: tabela com 97% de registros históricos (`aprovada`, `reprovada`) e 3% ativos (`pendente`). A tela operacional sempre busca os pendentes — faz sentido indexar só eles.

In [124]:
# Limpamos os índices anteriores
cur.execute("DROP INDEX IF EXISTS idx_comp_status_created")
cur.execute("DROP INDEX IF EXISTS idx_comp_created_status")

# Índice completo em status
cur.execute("CREATE INDEX idx_full_status ON propostas (status)")

# Índice parcial: apenas propostas pendentes
cur.execute("""
    CREATE INDEX idx_partial_pendente
    ON propostas (created_at)
    WHERE status = 'pendente'
""")

print("Índices criados.")

Índices criados.


In [125]:
# Comparação de tamanho: índice completo vs parcial
cur.execute("""
    SELECT
        indexname,
        pg_size_pretty(pg_relation_size(indexname::regclass)) AS tamanho
    FROM pg_indexes
    WHERE tablename = 'propostas'
      AND indexname IN ('idx_full_status', 'idx_partial_pendente')
    ORDER BY indexname
""")

print(f"{'Índice':<30} {'Tamanho':>10}")
print("-" * 42)
for row in cur.fetchall():
    print(f"{row['indexname']:<30} {row['tamanho']:>10}")

Índice                            Tamanho
------------------------------------------
idx_full_status                    360 kB
idx_partial_pendente                48 kB


In [126]:
# Query na tela de fila — usa o índice parcial automaticamente
# O planner reconhece que o predicado WHERE status = 'pendente' coincide com o índice parcial
SQL_FILA = """
SELECT id, numero, valor, created_at
FROM propostas
WHERE status = 'pendente'
ORDER BY created_at DESC
LIMIT 50
"""

print("=== Query da fila de pendentes ===")
print("Esperado: Index Scan using idx_partial_pendente")
explain(SQL_FILA)

=== Query da fila de pendentes ===
Esperado: Index Scan using idx_partial_pendente
Limit  (cost=0.28..163.43 rows=50 width=86)
  ->  Index Scan Backward using idx_partial_pendente on propostas  (cost=0.28..816.03 rows=250 width=86)


In [127]:
# Query fora do predicado do índice parcial — usa índice completo ou Seq Scan
SQL_APROVADAS = """
SELECT id, numero
FROM propostas
WHERE status = 'aprovada'
LIMIT 50
"""

print("=== Query para status = 'aprovada' ===")
print("O índice parcial não é usado (predicado não bate)")
explain(SQL_APROVADAS)

=== Query para status = 'aprovada' ===
O índice parcial não é usado (predicado não bate)
Limit  (cost=6.23..92.02 rows=50 width=62)
  ->  Bitmap Heap Scan on propostas  (cost=6.23..435.17 rows=250 width=62)
        Recheck Cond: ((status)::text = 'aprovada'::text)
        ->  Bitmap Index Scan on idx_full_status  (cost=0.00..6.17 rows=250 width=0)
              Index Cond: ((status)::text = 'aprovada'::text)


## 4. Índice Funcional — Indexar Expressões

Quando a query aplica uma função sobre uma coluna, o índice simples nessa coluna não é usado. É preciso criar um índice sobre a expressão.

Casos comuns:
- `LOWER(nome)` para busca case-insensitive
- `DATE(created_at)` para agrupar por dia
- `(response_data->>'score')::int` para filtrar campo JSONB como inteiro

In [128]:
# Sem índice funcional — LOWER(nome) força Seq Scan mesmo com índice em nome
cur.execute("CREATE INDEX IF NOT EXISTS idx_clientes_nome ON clientes (nome)")

SQL_LOWER = """
SELECT id, nome
FROM clientes
WHERE LOWER(nome) = LOWER('cliente 00042')
"""

print("=== Sem índice funcional (LOWER) ===")
explain(SQL_LOWER)

=== Sem índice funcional (LOWER) ===
Seq Scan on clientes  (cost=0.00..244.00 rows=50 width=422)
  Filter: (lower((nome)::text) = 'cliente 00042'::text)


In [129]:
# Criamos o índice funcional
cur.execute("""
    CREATE INDEX idx_clientes_nome_lower
    ON clientes (LOWER(nome))
""")

print("=== Com índice funcional LOWER(nome) ===")
explain(SQL_LOWER)

=== Com índice funcional LOWER(nome) ===
Bitmap Heap Scan on clientes  (cost=4.67..87.14 rows=50 width=422)
  Recheck Cond: (lower((nome)::text) = 'cliente 00042'::text)
  ->  Bitmap Index Scan on idx_clientes_nome_lower  (cost=0.00..4.66 rows=50 width=0)
        Index Cond: (lower((nome)::text) = 'cliente 00042'::text)


In [130]:
# Índice funcional em DATE(created_at) para queries por dia
cur.execute("""
    CREATE INDEX idx_propostas_data
    ON propostas (DATE(created_at))
""")

SQL_DATE = """
SELECT COUNT(*)
FROM propostas
WHERE DATE(created_at) = '2023-07-15'
"""

print("=== Índice funcional DATE(created_at) ===")
explain(SQL_DATE)

=== Índice funcional DATE(created_at) ===
Aggregate  (cost=436.42..436.43 rows=1 width=8)
  ->  Bitmap Heap Scan on propostas  (cost=6.23..435.79 rows=250 width=0)
        Recheck Cond: (date(created_at) = '2023-07-15'::date)
        ->  Bitmap Index Scan on idx_propostas_data  (cost=0.00..6.17 rows=250 width=0)
              Index Cond: (date(created_at) = '2023-07-15'::date)


In [131]:
# Populamos consultas_externas com dados JSONB
import json

cur.execute("SELECT id FROM propostas LIMIT 5000")
prop_ids = [row["id"] for row in cur.fetchall()]

consultas_data = []
for prop_id in prop_ids:
    score = random.randint(200, 950)
    aprovado = score >= 600
    consultas_data.append((
        prop_id,
        random.choice(["serasa", "spc", "boa_vista"]),
        json.dumps({
            "score": score,
            "status": "aprovado" if aprovado else "reprovado",
            "restricoes": random.randint(0, 5),
        }),
        "processado",
        datetime.now() - timedelta(days=random.randint(0, 365)),
    ))

psycopg2.extras.execute_values(
    cur,
    "INSERT INTO consultas_externas (proposta_id, provider, response_data, status, created_at) VALUES %s",
    consultas_data,
)
print(f"Consultas externas inseridas: {len(consultas_data)}")

Consultas externas inseridas: 5000


In [132]:
# Índice funcional em campo JSONB extraído como inteiro
# Nota: expressões com cast (::int) precisam de parênteses extras
cur.execute("""
    CREATE INDEX idx_consultas_score_jsonb
    ON consultas_externas (((response_data->>'score')::int))
""")

SQL_JSONB_SCORE = """
SELECT id, proposta_id, response_data->>'score' AS score
FROM consultas_externas
WHERE (response_data->>'score')::int > 700
"""

print("=== Índice funcional em campo JSONB ===")
explain(SQL_JSONB_SCORE)

=== Índice funcional em campo JSONB ===
Bitmap Heap Scan on consultas_externas  (cost=29.20..156.71 rows=1667 width=40)
  Recheck Cond: (((response_data ->> 'score'::text))::integer > 700)
  ->  Bitmap Index Scan on idx_consultas_score_jsonb  (cost=0.00..28.78 rows=1667 width=0)
        Index Cond: (((response_data ->> 'score'::text))::integer > 700)


## 5. GIN Index — Queries em JSONB

O GIN (Generalized Inverted Index) é o tipo correto para:
- Operador de containment `@>`: verifica se o JSONB contém um subconjunto
- Operador de existência `?`: verifica se uma chave existe
- Full-text search
- Arrays (`@>`, `&&`)

**GIN vs B-Tree para JSONB:**
- GIN com `jsonb_ops` (padrão): suporta `@>`, `?`, `?|`, `?&`
- GIN com `jsonb_path_ops`: só suporta `@>`, mas é menor e mais rápido para esse operador
- B-Tree funcional: use quando filtrar por um campo específico com `=`, `>`, `<`

In [133]:
# Sem GIN — containment @> forca Seq Scan
SQL_GIN = """
SELECT id, proposta_id, response_data
FROM consultas_externas
WHERE response_data @> '{"status": "aprovado"}'
"""

print("=== Sem GIN index (@> operador) ===")
explain(SQL_GIN)

=== Sem GIN index (@> operador) ===
Seq Scan on consultas_externas  (cost=0.00..152.50 rows=50 width=40)
  Filter: (response_data @> '{"status": "aprovado"}'::jsonb)


In [134]:
# GIN com jsonb_ops (padrao) — suporta todos os operadores JSONB
cur.execute("""
    CREATE INDEX idx_consultas_gin_default
    ON consultas_externas USING GIN (response_data)
""")

print("=== Com GIN (jsonb_ops) ===")
explain(SQL_GIN)

=== Com GIN (jsonb_ops) ===
Bitmap Heap Scan on consultas_externas  (cost=13.24..93.87 rows=50 width=40)
  Recheck Cond: (response_data @> '{"status": "aprovado"}'::jsonb)
  ->  Bitmap Index Scan on idx_consultas_gin_default  (cost=0.00..13.23 rows=50 width=0)
        Index Cond: (response_data @> '{"status": "aprovado"}'::jsonb)


In [135]:
# GIN com jsonb_path_ops — menor, mais rápido para @>, mas só suporta @>
cur.execute("DROP INDEX IF EXISTS idx_consultas_gin_default")
cur.execute("""
    CREATE INDEX idx_consultas_gin_path
    ON consultas_externas USING GIN (response_data jsonb_path_ops)
""")

print("=== Com GIN (jsonb_path_ops) ===")
explain(SQL_GIN)

# Comparação de tamanho
cur.execute("""
    SELECT
        indexname,
        pg_size_pretty(pg_relation_size(indexname::regclass)) AS tamanho
    FROM pg_indexes
    WHERE tablename = 'consultas_externas'
    ORDER BY indexname
""")
print("\nÍndices em consultas_externas:")
for row in cur.fetchall():
    print(f"  {row['indexname']:<35} {row['tamanho']:>10}")

=== Com GIN (jsonb_path_ops) ===
Bitmap Heap Scan on consultas_externas  (cost=8.81..89.44 rows=50 width=40)
  Recheck Cond: (response_data @> '{"status": "aprovado"}'::jsonb)
  ->  Bitmap Index Scan on idx_consultas_gin_path  (cost=0.00..8.80 rows=50 width=0)
        Index Cond: (response_data @> '{"status": "aprovado"}'::jsonb)

Índices em consultas_externas:
  consultas_externas_pkey                 128 kB
  idx_consultas_gin_path                   96 kB
  idx_consultas_score_jsonb                72 kB


## 6. Caso Real — Tela de Fila de Propostas Pendentes Travando

**Cenário**: A tela operacional que lista a fila de propostas a analisar está lenta. O time relata que a consulta demora vários segundos mesmo retornando apenas 50 registros.

**Query problema**:
```sql
SELECT * FROM propostas
WHERE status = 'pendente'
ORDER BY created_at DESC
LIMIT 50
```

**Diagnóstico**: Seq Scan na tabela inteira + Sort em memória. O PostgreSQL lê 50.000 linhas para achar ~1.500 pendentes, ordena todas elas e retorna 50.

**Solução**: Índice parcial composto — indexa apenas os pendentes, já ordenados por `created_at`.

In [136]:
# Remove índices existentes em propostas para simular estado inicial
# Filtra PKs e UNIQUE constraints (não podem ser dropados como índice)
cur.execute("""
    SELECT i.indexname
    FROM pg_indexes i
    LEFT JOIN pg_constraint c
        ON c.conname = i.indexname
    WHERE i.tablename = 'propostas'
      AND i.indexname NOT LIKE '%pkey%'
      AND c.conname IS NULL
""")
existing = [row["indexname"] for row in cur.fetchall()]
for idx in existing:
    cur.execute(f"DROP INDEX IF EXISTS {idx}")
    print(f"Removido: {idx}")

print("\nEstado inicial: sem índices customizados.")

Removido: idx_full_status
Removido: idx_partial_pendente
Removido: idx_propostas_data

Estado inicial: sem índices customizados.


In [137]:
SQL_FILA_REAL = """
SELECT *
FROM propostas
WHERE status = 'pendente'
ORDER BY created_at DESC
LIMIT 50
"""

print("=== ANTES: Sem índice ===")
explain(SQL_FILA_REAL, analyze=True)
print()
print("Problema: Seq Scan em 50.000 linhas + Sort para encontrar 50 pendentes.")

=== ANTES: Sem índice ===
Limit  (cost=1144.30..1144.43 rows=50 width=152) (actual time=3.631..3.636 rows=50 loops=1)
  Buffers: shared hit=511
  ->  Sort  (cost=1144.30..1144.93 rows=250 width=152) (actual time=3.629..3.631 rows=50 loops=1)
        Sort Key: created_at DESC
        Sort Method: top-N heapsort  Memory: 33kB
        Buffers: shared hit=511
        ->  Seq Scan on propostas  (cost=0.00..1136.00 rows=250 width=152) (actual time=0.015..3.469 rows=1452 loops=1)
              Filter: ((status)::text = 'pendente'::text)
              Rows Removed by Filter: 48548
              Buffers: shared hit=511
Planning:
  Buffers: shared hit=5
Planning Time: 0.169 ms
Execution Time: 3.684 ms

Problema: Seq Scan em 50.000 linhas + Sort para encontrar 50 pendentes.


In [138]:
# Solução: índice parcial composto
# WHERE status = 'pendente' — restringe o índice aos dados quentes
# (created_at DESC) — já entrega na ordem que a query precisa
cur.execute("""
    CREATE INDEX idx_propostas_fila_pendente
    ON propostas (created_at DESC)
    WHERE status = 'pendente'
""")

print("=== DEPOIS: Com índice parcial ===")
explain(SQL_FILA_REAL, analyze=True)
print()
print("Resultado: Index Scan direto nos ~1.500 pendentes, retorna 50 imediatamente.")
print("Sem Sort — o índice já está ordenado por created_at DESC.")

=== DEPOIS: Com índice parcial ===
Limit  (cost=0.28..163.43 rows=50 width=152) (actual time=0.024..0.071 rows=50 loops=1)
  Buffers: shared hit=50 read=2
  ->  Index Scan using idx_propostas_fila_pendente on propostas  (cost=0.28..816.03 rows=250 width=152) (actual time=0.023..0.067 rows=50 loops=1)
        Buffers: shared hit=50 read=2
Planning:
  Buffers: shared hit=15 read=1
Planning Time: 0.259 ms
Execution Time: 0.093 ms

Resultado: Index Scan direto nos ~1.500 pendentes, retorna 50 imediatamente.
Sem Sort — o índice já está ordenado por created_at DESC.


In [139]:
# Métricas: quantas linhas o índice precisa manter?
cur.execute("""
    SELECT
        COUNT(*) FILTER (WHERE status = 'pendente') AS pendentes,
        COUNT(*) AS total,
        ROUND(
            COUNT(*) FILTER (WHERE status = 'pendente') * 100.0 / COUNT(*), 2
        ) AS pct_pendente,
        pg_size_pretty(pg_relation_size('idx_propostas_fila_pendente')) AS tamanho_indice,
        pg_size_pretty(pg_relation_size('propostas')) AS tamanho_tabela
    FROM propostas
""")

row = cur.fetchone()
print(f"Pendentes  : {row['pendentes']:,} ({row['pct_pendente']}% do total)")
print(f"Total      : {row['total']:,}")
print(f"Índice     : {row['tamanho_indice']} (vs tabela: {row['tamanho_tabela']})")

Pendentes  : 1,452 (2.90% do total)
Total      : 50,000
Índice     : 48 kB (vs tabela: 4088 kB)


## 7. Anti-Patterns de Indexação

Cada índice custa INSERT, UPDATE e DELETE. Índices desnecessários desperdiçam disco, aumentam WAL e confundem o planner.

**Índice nunca usado** — Consulte `pg_stat_user_indexes.idx_scan` após semanas de produção. `idx_scan = 0` em índice com meses de vida: candidato a remoção. Exceção: índices que existem para UNIQUE constraint ou que cobrem queries raras mas críticas (ex: auditoria anual).

**Índice redundante** — `(status)` junto com `(status, created_at)`: o composto já cobre queries que filtram só por `status`. O simples é peso morto.

**10+ índices numa tabela OLTP de alta escrita** — Cada INSERT precisa atualizar todos os índices. Em tabelas com milhares de writes/segundo, isso vira gargalo. Priorize: cubra as queries do hot path e remova o resto.

**Indexar coluna de baixa cardinalidade sem predicado parcial.** `WHERE ativo = true` numa tabela onde 90% é `true` — o planner sabe que Seq Scan é mais barato. Se precisa mesmo desse filtro, use índice parcial `WHERE ativo = false` (indexa os 10% que interessam).

In [140]:
# Criamos índices desnecessários para demonstração
cur.execute("CREATE INDEX idx_dup_status          ON propostas (status)")
cur.execute("CREATE INDEX idx_dup_status_created  ON propostas (status, created_at)")
cur.execute("CREATE INDEX idx_dup_status_valor    ON propostas (status, valor)")
cur.execute("CREATE INDEX idx_dup_valor           ON propostas (valor)")

print("Índices duplicados/excessivos criados para demonstração.")

# Forçar coleta de estatísticas
cur.execute("ANALYZE propostas")

# Verificar todos os índices e seus tamanhos
cur.execute("""
    SELECT
        i.indexname,
        pg_size_pretty(pg_relation_size(i.indexname::regclass)) AS tamanho,
        s.idx_scan,
        s.idx_tup_read,
        s.idx_tup_fetch
    FROM pg_indexes i
    JOIN pg_stat_user_indexes s
        ON s.indexrelname = i.indexname AND s.schemaname = i.schemaname
    WHERE i.tablename = 'propostas'
    ORDER BY pg_relation_size(i.indexname::regclass) DESC
""")

print(f"\n{'Índice':<35} {'Tamanho':>8}  {'Scans':>6}")
print("-" * 55)
for row in cur.fetchall():
    # Não flagar PKs nem UNIQUE constraints como "nunca usado"
    is_constraint = "pkey" in row["indexname"] or "_key" in row["indexname"]
    alerta = " <- NUNCA USADO" if row["idx_scan"] == 0 and not is_constraint else ""
    print(f"{row['indexname']:<35} {row['tamanho']:>8}  {row['idx_scan']:>6}{alerta}")

Índices duplicados/excessivos criados para demonstração.

Índice                               Tamanho   Scans
-------------------------------------------------------
idx_dup_status_valor                 1960 kB       0 <- NUNCA USADO
propostas_numero_key                 1552 kB       0
idx_dup_valor                        1464 kB       0 <- NUNCA USADO
propostas_pkey                       1112 kB  205000
idx_dup_status_created                552 kB       0 <- NUNCA USADO
idx_dup_status                        360 kB       0 <- NUNCA USADO
idx_propostas_fila_pendente            48 kB       1


In [141]:
# Identificar índices redundantes:
# idx_dup_status é coberto por idx_dup_status_created (leftmost prefix)
# Qualquer query que usaria (status) pode usar (status, created_at)

cur.execute("""
    SELECT
        indexname,
        indexdef
    FROM pg_indexes
    WHERE tablename = 'propostas'
      AND indexname NOT LIKE '%pkey%'
    ORDER BY indexname
""")

print("Índices ativos e suas definições:")
for row in cur.fetchall():
    print(f"  {row['indexname']}")
    print(f"    {row['indexdef']}")

print()
print("Análise:")
print("  idx_dup_status (status) é coberto por idx_dup_status_created (status, created_at)")
print("  idx_dup_status pode ser removido sem perda de cobertura.")

Índices ativos e suas definições:
  idx_dup_status
    CREATE INDEX idx_dup_status ON public.propostas USING btree (status)
  idx_dup_status_created
    CREATE INDEX idx_dup_status_created ON public.propostas USING btree (status, created_at)
  idx_dup_status_valor
    CREATE INDEX idx_dup_status_valor ON public.propostas USING btree (status, valor)
  idx_dup_valor
    CREATE INDEX idx_dup_valor ON public.propostas USING btree (valor)
  idx_propostas_fila_pendente
    CREATE INDEX idx_propostas_fila_pendente ON public.propostas USING btree (created_at DESC) WHERE ((status)::text = 'pendente'::text)
  propostas_numero_key
    CREATE UNIQUE INDEX propostas_numero_key ON public.propostas USING btree (numero)

Análise:
  idx_dup_status (status) é coberto por idx_dup_status_created (status, created_at)
  idx_dup_status pode ser removido sem perda de cobertura.


In [142]:
# Custo de escrita: comparar tempo de INSERT com muitos vs poucos índices
import time

def benchmark_insert(n=1000, label=""):
    cur.execute("SELECT MAX(id) FROM clientes")
    max_id = cur.fetchone()["max"]

    batch = [
        (
            f"BENCH-{i:08d}",
            random.randint(1, max_id),
            round(random.uniform(500, 50000), 2),
            random.choice(["pendente", "aprovada", "reprovada"]),
            random.randint(300, 900),
            datetime.now(),
        )
        for i in range(n)
    ]

    # Desliga autocommit temporariamente para usar transação
    conn.autocommit = False
    start = time.perf_counter()
    psycopg2.extras.execute_values(
        cur,
        "INSERT INTO propostas (numero, cliente_id, valor, status, score, created_at) VALUES %s",
        batch,
    )
    elapsed = time.perf_counter() - start
    conn.rollback()  # Desfaz os inserts de benchmark
    conn.autocommit = True

    print(f"{label}: {n} INSERTs em {elapsed:.3f}s ({n/elapsed:.0f} ops/s)")

print("Com 4 índices extras:")
benchmark_insert(2000, "  (muitos índices)")

# Remove índices desnecessários
for idx in ["idx_dup_status", "idx_dup_status_created", "idx_dup_status_valor", "idx_dup_valor"]:
    cur.execute(f"DROP INDEX IF EXISTS {idx}")

print("\nApós remover 4 índices desnecessários:")
benchmark_insert(2000, "  (índices mínimos)")

Com 4 índices extras:
  (muitos índices): 2000 INSERTs em 0.086s (23205 ops/s)

Após remover 4 índices desnecessários:
  (índices mínimos): 2000 INSERTs em 0.063s (31744 ops/s)


## Cheat Sheet

| Tipo | Operadores | Quando usar |
|------|-----------|--------------------|
| B-Tree | `=`, `<`, `>`, `BETWEEN`, `LIKE 'x%'` | Default. Cobre campos escalares |
| B-Tree composto | Leftmost prefix | Múltiplos filtros. Igualdade antes de range |
| Parcial | Qualquer + `WHERE` | Subset quente (3-10% da tabela) |
| Funcional | Expressão (`LOWER()`, `DATE()`) | Query aplica função na coluna |
| GIN (`jsonb_ops`) | `@>`, `?`, `?|`, `?&` | JSONB genérico, arrays, full-text |
| GIN (`jsonb_path_ops`) | Apenas `@>` | Containment em JSONB, ~30% menor que `jsonb_ops` |

**Decisões rápidas:**

- Composto `(A, B)` cobre queries em `A` sozinho. Índice simples em `A` é redundante — remova.
- 97% dos registros são históricos? Índice parcial no subset ativo. Índice full é desperdício.
- Query usa `LOWER(col)` ou `DATE(col)`? Índice em `col` não ajuda. Crie índice funcional na expressão exata.
- `pg_stat_user_indexes.idx_scan = 0` depois de semanas? Remova, a menos que seja UNIQUE ou cobertura de query rara crítica.
- Tabela OLTP com alta escrita e 10+ índices? Algo está errado. Priorize hot path e corte o resto.

## Exercícios

**Schema de referência:**

```sql
propostas      (id, numero, cliente_id, valor, status, score, created_at)
clientes       (id, cpf, nome, renda_mensal, created_at)
consultas_externas (id, proposta_id, provider, response_data JSONB, status, created_at)
decisoes_log   (id, proposta_id, etapa, resultado, regras_aplicadas JSONB, executed_at)
parcelas       (id, proposta_id, numero, valor, vencimento, status_pagamento, pago_em)
```

**Exercício 1 — Tipo de índice por query**

Para cada query, defina o tipo de índice e justifique:

```sql
-- a)
SELECT * FROM clientes WHERE LOWER(cpf) = '12345678900';

-- b)
SELECT * FROM consultas_externas
WHERE response_data @> '{"provider": "serasa", "restricoes": 0}';

-- c)
SELECT * FROM parcelas
WHERE vencimento BETWEEN '2024-01-01' AND '2024-03-31'
  AND status_pagamento = 'pendente';

-- d)
SELECT * FROM propostas
WHERE score > 700
ORDER BY created_at DESC
LIMIT 20;
```

**Exercício 2 — Índice parcial**

A tabela `parcelas` tem ~200k linhas. Aproximadamente 50% estão com `status_pagamento = 'pendente'`. O sistema consulta parcelas pendentes vencidas todo dia:

```sql
SELECT proposta_id, vencimento
FROM parcelas
WHERE status_pagamento = 'pendente'
  AND vencimento < CURRENT_DATE
ORDER BY vencimento ASC;
```

Qual `CREATE INDEX` cobre essa query?

**Exercício 3 — Ordem em índice composto**

Três queries frequentes em `decisoes_log`:

```sql
-- Query A: filtra por etapa e ordena por data
SELECT * FROM decisoes_log
WHERE etapa = 'score'
ORDER BY executed_at DESC;

-- Query B: filtra por etapa e resultado
SELECT * FROM decisoes_log
WHERE etapa = 'score' AND resultado = 'aprovado';

-- Query C: filtra só por resultado
SELECT * FROM decisoes_log
WHERE resultado = 'reprovado';
```

Com no máximo dois índices, cubra as três queries. Mostre quais índices e explique o mapeamento.

**Exercício 4 — Diagnóstico de índices não usados**

Interprete o resultado da query abaixo e decida o que manter/remover:

```sql
SELECT
    schemaname,
    tablename,
    indexname,
    idx_scan,
    pg_size_pretty(pg_relation_size(indexrelid)) AS tamanho
FROM pg_stat_user_indexes
WHERE idx_scan = 0
  AND indexname NOT LIKE '%pkey%'
ORDER BY pg_relation_size(indexrelid) DESC;
```

**Exercício 5 — GIN vs B-Tree funcional**

Duas queries em `consultas_externas.response_data` (JSONB):

```sql
-- Query X: containment
SELECT * FROM consultas_externas
WHERE response_data @> '{"status": "aprovado"}';

-- Query Y: comparação numérica
SELECT * FROM consultas_externas
WHERE (response_data->>'score')::int > 750;
```

Um único GIN resolve ambas? Explique e proponha a estratégia correta.

## Respostas

---

### Exercício 1

**a)** Índice funcional em `LOWER(cpf)`:

```sql
CREATE INDEX idx_clientes_cpf_lower ON clientes (LOWER(cpf));
```

CPF é VARCHAR, não numérico — `LOWER()` na query invalida qualquer índice direto em `cpf`. O índice funcional casa com a expressão exata do WHERE.

Na prática, CPF raramente precisa de case-insensitive (são dígitos). Se a query puder ser reescrita sem `LOWER()`, um B-Tree simples em `cpf` basta e já existe (UNIQUE constraint).

**b)** GIN com `jsonb_ops` ou `jsonb_path_ops`:

```sql
-- jsonb_path_ops: menor e mais rápido para @>, suficiente aqui
CREATE INDEX idx_consultas_response_gin
    ON consultas_externas USING GIN (response_data jsonb_path_ops);
```

O operador `@>` (containment) exige GIN. B-Tree não suporta. `jsonb_path_ops` é a melhor escolha quando só se usa `@>` — ocupa ~30% menos espaço que `jsonb_ops`.

Se no futuro precisar de `?` (existência de chave), terá que trocar para `jsonb_ops`.

**c)** B-Tree composto `(status_pagamento, vencimento)`:

```sql
CREATE INDEX idx_parcelas_status_vencimento
    ON parcelas (status_pagamento, vencimento);
```

`status_pagamento = 'pendente'` é igualdade (vai primeiro), `vencimento BETWEEN` é range (vai depois). O B-Tree navega até a folha de `'pendente'` e percorre sequencialmente o range de datas.

Alternativa: índice parcial `ON parcelas (vencimento) WHERE status_pagamento = 'pendente'`. Menor, mas só serve para essa query específica.

**d)** B-Tree composto `(score, created_at DESC)`:

```sql
CREATE INDEX idx_propostas_score_created
    ON propostas (score, created_at DESC);
```

`score > 700` é range (não igualdade), então a direção do ORDER BY precisa estar no índice para evitar Sort. O `DESC` no `created_at` casa com o `ORDER BY created_at DESC`.

Gotcha: com `score > 700` retornando ~22% da tabela (700 de 300-900), o planner pode ignorar o índice. Se isso acontecer, considere índice parcial `WHERE score > 700` — mas só se essa query for frequente o suficiente para justificar.

---

### Exercício 2

```sql
CREATE INDEX idx_parcelas_pendente_vencimento
    ON parcelas (vencimento ASC)
    WHERE status_pagamento = 'pendente';
```

Índice parcial: indexa só as parcelas pendentes (~50% da tabela), já ordenado por `vencimento ASC` para casar com o `ORDER BY`. O predicado `vencimento < CURRENT_DATE` faz um range scan eficiente nas folhas.

Por que parcial e não composto? Neste caso a seletividade de `status_pagamento` é 50%, então o ganho do parcial vs composto é moderado. Mas o índice parcial tem metade do tamanho e metade do custo de manutenção em INSERTs — parcelas pagas nunca tocam esse índice.

Se a distribuição mudar (ex: 95% pago, 5% pendente), o ganho do parcial fica muito maior.

---

### Exercício 3

Dois índices:

```sql
-- Índice 1: cobre Query A (etapa + ORDER BY) e Query B (etapa + resultado)
CREATE INDEX idx_decisoes_etapa_executed
    ON decisoes_log (etapa, executed_at DESC);

-- Índice 2: cobre Query C (resultado sozinho)
CREATE INDEX idx_decisoes_resultado
    ON decisoes_log (resultado);
```

Mapeamento:
- **Query A** (`WHERE etapa = 'score' ORDER BY executed_at DESC`): usa `idx_decisoes_etapa_executed`. Igualdade em `etapa` (leftmost), depois percorre folhas já ordenadas por `executed_at DESC`. Sem Sort no plano.
- **Query B** (`WHERE etapa = 'score' AND resultado = 'aprovado'`): usa `idx_decisoes_etapa_executed` para filtrar por `etapa`, depois aplica Filter em `resultado`. Não é perfeito (Filter no heap), mas com `etapa` já reduzindo bastante, é aceitável.
- **Query C** (`WHERE resultado = 'reprovado'`): usa `idx_decisoes_resultado`. Não pode usar o índice 1 porque `resultado` não é leftmost.

Alternativa: trocar índice 1 por `(etapa, resultado, executed_at DESC)`. Cobre A e B sem Filter, mas o índice fica maior. Depende da frequência de cada query.

---

### Exercício 4

O que remover:
- Índices grandes com `idx_scan = 0` que não são UNIQUE constraint e não cobrem queries raras críticas. Se o índice existe há semanas e nunca foi usado, ele consome disco e torna writes mais lentos sem retorno.

O que manter mesmo com `idx_scan = 0`:
- Índices de UNIQUE constraint (não aparecem na query por causa do filtro `NOT LIKE '%pkey%'`, mas UNIQUE constraints fora de PK podem aparecer)
- Índices criados recentemente (ainda não tiveram tempo de ser usados)
- Índices que cobrem queries de auditoria/compliance que rodam uma vez por mês/ano
- Índices parciais muito pequenos (custo de manutenção desprezível)

Antes de remover em produção: confirme com o time que nenhuma query sazonal depende do índice. Use `DROP INDEX CONCURRENTLY` para não bloquear a tabela.

---

### Exercício 5

Não. Um único GIN não resolve as duas.

**Query X** (`@>` containment): GIN resolve. Tanto `jsonb_ops` quanto `jsonb_path_ops` suportam `@>`.

**Query Y** (`(response_data->>'score')::int > 750`): GIN não serve. O operador `>` em valor extraído e castado é uma comparação escalar, não um operador JSONB. GIN não suporta range queries em valores extraídos.

Estratégia correta — dois índices:

```sql
-- Para Query X: GIN com jsonb_path_ops (menor, só precisa de @>)
CREATE INDEX idx_consultas_gin
    ON consultas_externas USING GIN (response_data jsonb_path_ops);

-- Para Query Y: B-Tree funcional na expressão exata
CREATE INDEX idx_consultas_score_func
    ON consultas_externas ((response_data->>'score')::int);
```

O B-Tree funcional precisa casar exatamente com a expressão no WHERE. Se a query usar `(response_data->'score')::int` (com `->` em vez de `->>`), o índice não será usado.